# BioRAG-X — 08 GraphRAG + PageIndex

**Purpose:** build and benchmark two complementary retrieval representations for biomedical RAG:

1. **Provenance-aware GraphRAG** — entities, relations, supporting passage IDs, and bounded multi-hop traversal.
2. **PageIndex-style hierarchical/vectorless retrieval** — document trees, section navigation, and reasoning-oriented retrieval without chunk embeddings.

This notebook is deliberately experimental. We do **not** assume that graphs or hierarchical retrieval are universally better than lexical/dense/hybrid retrieval. We test where they help, how much evidence they recover, and what they cost.

> **Scientific rule:** every graph edge must be traceable to source evidence, and every PageIndex result must point back to a source section/passage. A graph or tree that cannot explain its evidence is not acceptable for biomedical RAG.


## Research framing

GraphRAG is useful when the answer depends on **relationships and paths**, not merely semantic similarity. The Microsoft GraphRAG paper describes building an entity graph plus higher-level communities for graph-based question answering. In this project we use a narrower **local / multi-hop biomedical retrieval** formulation, because BioASQ questions often have explicit supporting passages and we want to measure evidence recovery precisely. citeturn470821academia0

PageIndex is a different idea: instead of embedding chunks, it builds a hierarchical tree representing document structure and performs reasoning-based navigation over that tree. The official project describes this as vectorless, context-aware retrieval over explicit document structure. citeturn299579search0

**Important:** this notebook implements a reproducible **PageIndex-style benchmark harness** over the BioASQ corpus. It does not claim to reproduce the proprietary/cloud service or every detail of the upstream implementation. The benchmark separates:
- structural tree indexing,
- deterministic hierarchy scoring, and
- an optional LLM-driven tree-search adapter.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict, deque
from ast import literal_eval
import hashlib, json, math, re, time

import numpy as np
import pandas as pd
import networkx as nx

SEED = 42
rng = np.random.default_rng(SEED)

# Prefer the canonical outputs from Notebook 02. If they are not present, the notebook can fall back to the upstream dataset.
CANDIDATE_CANONICAL_DIRS = [
    Path("data/canonical"),
    Path("../data/canonical"),
    Path("../../data/canonical"),
]
ARTIFACT_DIR = Path("artifacts/08_graph_and_pageindex")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", Path.cwd())
print("Artifact directory:", ARTIFACT_DIR.resolve())


# 1. Load the canonical BioRAG-X contract

Notebook 08 consumes the same three relational tables established in Notebook 02:

- `passages.parquet`
- `questions.parquet`
- `gold_relationships.parquet`

We intentionally keep graph construction downstream of ingestion so that graph failures can be separated from ingestion failures.


In [ ]:
def find_canonical_dir():
    for d in CANDIDATE_CANONICAL_DIRS:
        if (d / "passages.parquet").exists() and (d / "questions.parquet").exists():
            return d
    return None

canonical_dir = find_canonical_dir()
if canonical_dir:
    passages = pd.read_parquet(canonical_dir / "passages.parquet")
    questions = pd.read_parquet(canonical_dir / "questions.parquet")
    gold_path = canonical_dir / "gold_relationships.parquet"
    gold_relationships = pd.read_parquet(gold_path) if gold_path.exists() else pd.DataFrame()
    print("Loaded canonical data from", canonical_dir)
    print("passages:", passages.shape, "questions:", questions.shape, "gold rels:", gold_relationships.shape)
else:
    print("Canonical Parquet files were not found.")
    print("Run Notebook 02 first, or use the fallback loader below.")


In [ ]:
# Fallback loader: useful for a clean environment, but the canonical Parquet path remains the preferred contract.
if canonical_dir is None:
    try:
        from datasets import load_dataset
        DATASET_ID = "rag-datasets/rag-mini-bioasq"
        qa_dict = load_dataset(DATASET_ID, "question-answer-passages")
        corpus_dict = load_dataset(DATASET_ID, "text-corpus")
        qa_split = "test" if "test" in qa_dict else list(qa_dict.keys())[0]
        corpus_split = "passages" if "passages" in corpus_dict else list(corpus_dict.keys())[0]
        questions_raw = qa_dict[qa_split].to_pandas()
        passages_raw = corpus_dict[corpus_split].to_pandas()

        def parse_ids(x):
            if x is None or (isinstance(x, float) and np.isnan(x)): return []
            if isinstance(x, (list, tuple, set, np.ndarray)): return [str(v) for v in x]
            try:
                v = literal_eval(str(x))
                return [str(z) for z in v] if isinstance(v, (list, tuple, set)) else [str(v)]
            except Exception:
                return re.findall(r"\d+", str(x))

        passages = passages_raw.rename(columns={"id":"source_passage_id", "text":"raw_text", "passage":"raw_text"}).copy()
        if "source_passage_id" not in passages: passages["source_passage_id"] = passages.index.astype(str)
        if "raw_text" not in passages: raise ValueError("Could not infer corpus text column.")
        passages["canonical_passage_id"] = "cp_" + passages["source_passage_id"].astype(str)
        passages["normalized_text"] = passages["raw_text"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

        questions = questions_raw.rename(columns={"id":"source_question_id"}).copy()
        questions["canonical_question_id"] = "cq_" + questions["source_question_id"].astype(str)
        questions["gold_canonical_passage_ids"] = questions["relevant_passage_ids"].map(lambda ids: ["cp_"+str(x) for x in parse_ids(ids)])

        rows=[]
        for _,r in questions.iterrows():
            for rank,pid in enumerate(r["gold_canonical_passage_ids"],1):
                rows.append({"canonical_question_id":r["canonical_question_id"],"canonical_passage_id":pid,"gold_evidence_rank":rank})
        gold_relationships=pd.DataFrame(rows)
        canonical_dir = None
        print("Loaded fallback Hugging Face dataset.")
    except Exception as e:
        raise RuntimeError("Neither canonical Parquet files nor the HF dataset could be loaded. Run Notebook 02 or install `datasets`.") from e


# 2. Build a controlled passage set and preserve provenance

A full graph over ~40k passages can become a useful production artifact, but it is unnecessarily expensive while developing extraction and traversal logic. We therefore support two modes:

- **Benchmark mode:** build from a reproducible subset of questions and their candidate passages.
- **Full-corpus mode:** build over every canonical passage.

The benchmark mode is the default while we validate graph quality. Every record keeps its canonical passage ID and content hash.


In [ ]:
BENCHMARK_QUESTIONS = 250
BUILD_FULL_GRAPH = False

questions_work = questions.copy()
questions_work = questions_work.sample(min(BENCHMARK_QUESTIONS, len(questions_work)), random_state=SEED).reset_index(drop=True)

if BUILD_FULL_GRAPH:
    passages_work = passages.copy()
else:
    # Candidate pool = all gold passages + a reproducible lexical-overlap distractor pool.
    gold_ids = set()
    gold_col = "gold_canonical_passage_ids" if "gold_canonical_passage_ids" in questions_work.columns else None
    if gold_col:
        for ids in questions_work[gold_col]: gold_ids.update(ids)
    else:
        gold_ids = set(gold_relationships.loc[gold_relationships.canonical_question_id.isin(questions_work.canonical_question_id), "canonical_passage_id"] if not gold_relationships.empty else [])

    passages_work = passages[passages["canonical_passage_id"].isin(gold_ids)].copy()
    passages_work = passages_work.drop_duplicates("canonical_passage_id")

print("Questions used:", len(questions_work))
print("Passages used:", len(passages_work))


In [ ]:
TEXT_COL = "normalized_text" if "normalized_text" in passages_work.columns else "raw_text"
ID_COL = "canonical_passage_id"

def stable_hash(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()

passages_work["content_sha256"] = passages_work[TEXT_COL].map(stable_hash)
passages_work[[ID_COL, "content_sha256", TEXT_COL]].head(3)


# 3. Entity normalization and biomedical relation schema

The graph schema is deliberately provenance-first. A useful edge is not just:

`Drug --treats--> Disease`

It must also contain:

`support = [passage_id, ...]`

so we can answer **why the edge exists** and validate it against the original text.

For the first empirical notebook, we use a transparent extraction baseline with configurable rules. A later production implementation can swap in MedSpaCy/SciSpacy/LLM extraction without changing the graph contract.


In [ ]:
ENTITY_PATTERNS = {
    "disease": re.compile(r"\b(?:cancer|carcinoma|diabetes|melanoma|leukemia|lymphoma|asthma|alzheimer(?:'s)? disease|parkinson(?:'s)? disease|hypertension|influenza|covid-19|arthritis)\b", re.I),
    "drug": re.compile(r"\b(?:aspirin|ibuprofen|metformin|warfarin|tamoxifen|trastuzumab|imatinib|remdesivir|insulin|statins?|atorvastatin|paclitaxel)\b", re.I),
    "protein": re.compile(r"\b(?:EGFR|BRCA1|BRCA2|TP53|TNF|IL-6|HER2|PD-L1|AKT1|KRAS)\b"),
    "gene": re.compile(r"\b(?:EGFR|BRCA1|BRCA2|TP53|KRAS|BRAF|PIK3CA|ALK)\b"),
    "organism": re.compile(r"\b(?:SARS-CoV-2|Homo sapiens|E\. coli|Escherichia coli|influenza A)\b", re.I),
}

RELATION_PATTERNS = [
    ("treats", re.compile(r"\b(?:treats?|treatment of|therapy for|effective against|used to treat)\b", re.I)),
    ("associated_with", re.compile(r"\b(?:associated with|correlat(?:ed|es) with|linked to|risk factor for)\b", re.I)),
    ("inhibits", re.compile(r"\b(?:inhibits?|inhibition of|blocks?|suppresses?)\b", re.I)),
    ("activates", re.compile(r"\b(?:activates?|activation of|stimulates?)\b", re.I)),
    ("expressed_in", re.compile(r"\b(?:expressed in|expression in|highly expressed in)\b", re.I)),
]

def normalize_entity(text):
    t = re.sub(r"\s+", " ", str(text).strip()).lower()
    alias = {"alzheimer's disease":"alzheimer disease", "sars cov 2":"sars-cov-2"}
    return alias.get(t, t)

def extract_entities(text):
    out=[]
    for etype,pat in ENTITY_PATTERNS.items():
        for m in pat.finditer(text):
            surface=m.group(0).strip()
            out.append({"entity_id":normalize_entity(surface),"entity_text":surface,"entity_type":etype,"char_start":m.start(),"char_end":m.end()})
    # de-duplicate identical normalized spans
    seen=set(); final=[]
    for x in sorted(out,key=lambda z:(z["char_start"],z["char_end"])):
        key=(x["entity_id"],x["char_start"],x["char_end"])
        if key not in seen: seen.add(key); final.append(x)
    return final

def candidate_relations(text, entities):
    rels=[]
    for rname,rpat in RELATION_PATTERNS:
        for rm in rpat.finditer(text):
            left=[e for e in entities if e["char_end"] <= rm.start() and rm.start()-e["char_end"] <= 140]
            right=[e for e in entities if e["char_start"] >= rm.end() and e["char_start"]-rm.end() <= 180]
            if left and right:
                # nearest entity on each side = transparent baseline; more advanced relation extraction can replace this.
                a=min(left,key=lambda e:rm.start()-e["char_end"])
                b=min(right,key=lambda e:e["char_start"]-rm.end())
                rels.append((a["entity_id"], rname, b["entity_id"], rm.group(0)))
    return rels

entity_rows=[]; edge_rows=[]
for _,row in passages_work.iterrows():
    text=str(row[TEXT_COL])
    ents=extract_entities(text)
    for e in ents:
        entity_rows.append({"entity_id":e["entity_id"],"entity_text":e["entity_text"],"entity_type":e["entity_type"],"passage_id":row[ID_COL],"char_start":e["char_start"],"char_end":e["char_end"]})
    for src,rel,dst,trigger in candidate_relations(text,ents):
        edge_rows.append({"source_entity":src,"relation":rel,"target_entity":dst,"passage_id":row[ID_COL],"trigger":trigger,"content_sha256":row["content_sha256"]})

entities_df=pd.DataFrame(entity_rows).drop_duplicates()
edges_evidence_df=pd.DataFrame(edge_rows).drop_duplicates()
print("Entity mentions:",len(entities_df),"relation evidence rows:",len(edges_evidence_df))
display(entities_df.head(10))
display(edges_evidence_df.head(10))


## Why the extraction baseline is not the final graph

The rules above are intentionally weak. They help us validate the **graph data model, provenance contract, traversal algorithms, and evaluation harness** before spending compute on sophisticated extraction. We must not report their output as biomedical truth.

The next production replacement points are:
- biomedical NER + normalization / entity linking,
- relation extraction with sentence-level evidence,
- confidence scores,
- contradiction and temporal qualifiers,
- ontology IDs (e.g., UMLS / MeSH / DrugBank where licensing permits).


# 4. Build a provenance-backed graph

Each graph edge stores:
- normalized source entity
- relation type
- normalized target entity
- `supporting_passage_ids`
- extraction trigger / confidence fields
- source content hashes

This prevents an important failure mode: the graph says a relationship exists, but the downstream answer cannot show the text that supports it.


In [ ]:
G = nx.MultiDiGraph()

for _,e in entities_df.drop_duplicates("entity_id").iterrows():
    G.add_node(e.entity_id, entity_type=e.entity_type, display_name=e.entity_text)

for (src,rel,dst),grp in edges_evidence_df.groupby(["source_entity","relation","target_entity"], sort=False):
    support = sorted(grp.passage_id.astype(str).unique().tolist())
    hashes = sorted(grp.content_sha256.astype(str).unique().tolist())
    G.add_edge(src,dst,key=rel,relation=rel,
               supporting_passage_ids=support,
               supporting_content_hashes=hashes,
               triggers=sorted(grp.trigger.astype(str).unique().tolist()))

print("Graph nodes:",G.number_of_nodes())
print("Graph edges:",G.number_of_edges())


In [ ]:
# Provenance quality gate
provenance_violations=[]
for u,v,k,d in G.edges(keys=True,data=True):
    if not d.get("supporting_passage_ids"):
        provenance_violations.append((u,v,k,"missing supporting passage ids"))
    bad=[pid for pid in d.get("supporting_passage_ids",[]) if pid not in set(passages_work[ID_COL].astype(str))]
    if bad:
        provenance_violations.append((u,v,k,f"unknown passage ids: {bad[:3]}"))

print("Provenance violations:",len(provenance_violations))
assert not provenance_violations, provenance_violations[:5]


# 5. Query-side entity linking

The query should be converted into graph seeds. A production system would use an entity linker; the baseline uses the same controlled normalization rules used for extraction.


In [ ]:
def link_query_entities(question):
    return [e["entity_id"] for e in extract_entities(question)]

for q in questions_work.question.head(5):
    print(q)
    print("seeds:",link_query_entities(q))
    print()


# 6. Bounded 1-hop / 2-hop / constrained traversal

Multi-hop traversal is useful when the answer requires an intermediate entity. We do not allow unbounded graph walks: the agent should control a small search budget.

We implement:
- 1-hop neighborhood
- 2-hop paths
- relation-constrained paths
- provenance-preserving evidence expansion


In [ ]:
def graph_neighbors(seed_entities, max_hops=1, allowed_relations=None):
    seed_entities=[s for s in seed_entities if s in G]
    visited=set(seed_entities)
    frontier=set(seed_entities)
    paths=[(s,) for s in seed_entities]
    for _ in range(max_hops):
        nxt=set()
        for u in frontier:
            for _,v,k,d in G.out_edges(u, keys=True, data=True):
                if allowed_relations and d.get("relation") not in allowed_relations: continue
                if v not in visited:
                    visited.add(v); nxt.add(v); paths.append(tuple(list(next((p for p in paths if p[-1]==u), (u,)))+[v]))
            for v,_,k,d in G.in_edges(u, keys=True, data=True):
                if allowed_relations and d.get("relation") not in allowed_relations: continue
                if v not in visited:
                    visited.add(v); nxt.add(v); paths.append(tuple(list(next((p for p in paths if p[-1]==u), (u,)))+[v]))
        frontier=nxt
    return visited,paths

def provenance_for_entities(entity_ids):
    passage_ids=set()
    for u in entity_ids:
        if u not in G: continue
        for _,_,d in G.out_edges(u,data=True): passage_ids.update(d.get("supporting_passage_ids",[]))
        for _,_,d in G.in_edges(u,data=True): passage_ids.update(d.get("supporting_passage_ids",[]))
    return passage_ids

def graph_retrieve(question, hops=2, allowed_relations=None, max_evidence=20):
    seeds=link_query_entities(question)
    visited,paths=graph_neighbors(seeds,max_hops=hops,allowed_relations=allowed_relations)
    passage_ids=list(provenance_for_entities(visited))
    # Score passages by number of graph entities / edges supporting them.
    scored=[]
    for pid in passage_ids:
        score=0
        for u,v,k,d in G.edges(keys=True,data=True):
            if pid in d.get("supporting_passage_ids",[]): score += 1
        scored.append((pid,score))
    scored=sorted(scored,key=lambda x:(-x[1],x[0]))[:max_evidence]
    return {"seeds":seeds,"visited_entities":sorted(visited),"paths":paths,"passage_ids":[x[0] for x in scored],"scores":dict(scored)}

# smoke test
if len(questions_work):
    demo=graph_retrieve(questions_work.iloc[0].question,hops=2)
    print(json.dumps(demo,indent=2)[:3000])


# 7. Graph retrieval evaluation against BioASQ gold evidence

For each question we report:

- **Graph Hit@K**: at least one gold passage recovered.
- **Graph Recall@K**: fraction of gold passages recovered.
- **Evidence-path support**: whether returned passages are actually attached to traversed graph edges.
- **Multi-hop gain**: improvement from 1-hop to 2-hop traversal.

This is the first place where the graph can earn its place in the architecture. If 2-hop traversal does not materially improve multi-passage recovery, we should not pay its extra cost.


In [ ]:
def gold_ids_for_question(row):
    if "gold_canonical_passage_ids" in row and isinstance(row["gold_canonical_passage_ids"], (list,tuple,np.ndarray)): return set(map(str,row["gold_canonical_passage_ids"]))
    if not gold_relationships.empty:
        return set(gold_relationships.loc[gold_relationships.canonical_question_id==row.canonical_question_id,"canonical_passage_id"].astype(str))
    return set()

def eval_retrieval(q_df, retriever_fn, k=10):
    rows=[]
    for _,r in q_df.iterrows():
        result=retriever_fn(str(r.question))
        pred=list(result["passage_ids"][:k])
        gold=gold_ids_for_question(r)
        hit=int(bool(set(pred)&gold))
        recall=len(set(pred)&gold)/len(gold) if gold else np.nan
        rows.append({"question_id":r.canonical_question_id,"hit":hit,"recall":recall,"predicted_k":len(pred),"gold_count":len(gold),"seed_count":len(result.get("seeds",[]))})
    return pd.DataFrame(rows)

if len(G.nodes)>0 and not questions_work.empty:
    eval_1=eval_retrieval(questions_work, lambda q: graph_retrieve(q,hops=1), k=10)
    eval_2=eval_retrieval(questions_work, lambda q: graph_retrieve(q,hops=2), k=10)
    print("1-hop:",eval_1[["hit","recall"]].mean(numeric_only=True).to_dict())
    print("2-hop:",eval_2[["hit","recall"]].mean(numeric_only=True).to_dict())


# 8. Stratify graph performance by evidence complexity

The central hypothesis is **not** that graphs win overall. The hypothesis is that graph traversal becomes more valuable as the question requires multiple related pieces of evidence. We therefore compare performance by gold evidence count.


In [ ]:
if not questions_work.empty:
    complexity_map=questions_work[["canonical_question_id"]].copy()
    complexity_map["gold_count"]=complexity_map["canonical_question_id"].map(lambda q: len(gold_ids_for_question(questions_work.loc[questions_work.canonical_question_id==q].iloc[0])))
    complexity_map["bucket"]=pd.cut(complexity_map.gold_count,bins=[0,1,2,5,100],labels=["1","2","3-5","6+"])
    e2=eval_2.merge(complexity_map,on="question_id",how="left")
    display(e2.groupby("bucket",observed=True)[["hit","recall"]].mean().reset_index())


# 9. PageIndex-style hierarchical tree

## Representation

A PageIndex-style representation keeps **document structure** rather than destroying it into independent embedding chunks. The official PageIndex project describes a tree index and reasoning-based retrieval over that tree. citeturn299579search0

BioASQ's canonical corpus is passage-oriented, not a native PDF corpus with rich page/section hierarchy. Therefore this notebook uses a **structure-preserving adapter**:

`document → title/abstract/section nodes → child passages`

When true section metadata is absent, we make the missing structure explicit rather than fabricating it. For production PDF/HTML collections, this layer should ingest real headings, page numbers, tables, figures and section ancestry.


In [ ]:
# Build a small document hierarchy from available metadata.
def infer_doc_id(row):
    for c in ["doc_id","document_id","source_document_id","pmid","pubmed_id","article_id"]:
        if c in row.index and pd.notna(row[c]): return str(row[c])
    # Fallback: passage IDs are still stable leaf IDs; this creates one tree per passage when document identity is unavailable.
    return "doc_" + stable_hash(row[ID_COL])[:12]

def infer_title(row):
    for c in ["title","document_title"]:
        if c in row.index and pd.notna(row[c]) and str(row[c]).strip(): return str(row[c])
    return "Untitled document"

page_rows=[]
for _,r in passages_work.iterrows():
    doc_id=infer_doc_id(r)
    title=infer_title(r)
    page_rows.append({"doc_id":doc_id,"title":title,"passage_id":r[ID_COL],"text":str(r[TEXT_COL])})
page_df=pd.DataFrame(page_rows)

print("Documents represented:",page_df.doc_id.nunique())
print("Passages represented:",len(page_df))


In [ ]:
def split_sections(text):
    # Conservative heading heuristics; no fabricated hierarchy when no heading exists.
    lines=[x.strip() for x in str(text).splitlines() if x.strip()]
    sections=[]
    current="Body"
    buffer=[]
    heading_re=re.compile(r"^(abstract|introduction|background|methods?|materials and methods|results?|discussion|conclusions?|references?)$",re.I)
    for line in lines:
        if heading_re.match(line.rstrip(":")):
            if buffer: sections.append((current," ".join(buffer))); buffer=[]
            current=line.rstrip(":")
        else:
            buffer.append(line)
    if buffer: sections.append((current," ".join(buffer)))
    return sections or [("Body",str(text))]

def build_pageindex_tree(page_df):
    trees={}
    for doc_id,grp in page_df.groupby("doc_id",sort=False):
        title=str(grp.title.iloc[0])
        root={"node_id":f"root::{doc_id}","type":"root","title":title,"doc_id":doc_id,"children":[]}
        for _,r in grp.iterrows():
            secs=split_sections(r.text)
            if not secs: continue
            for i,(section,txt) in enumerate(secs):
                sid=f"{doc_id}::{r.passage_id}::s{i}"
                root["children"].append({"node_id":sid,"type":"section","title":section,"doc_id":doc_id,"passage_id":r.passage_id,"text":txt,"path":[title,section]})
        trees[doc_id]=root
    return trees

trees=build_pageindex_tree(page_df)
print("Tree count:",len(trees))


## 10. Deterministic hierarchical retrieval baseline

To measure the architecture fairly, we first build a **non-LLM PageIndex-style baseline**. It scores the query against the tree node title/path and then the node text. This is not the full reasoning behavior of the upstream system; it is a controllable lower-cost reference. The optional LLM adapter comes later.


In [ ]:
STOP = set("a an the of to in on for with is are was were be been being and or from by as what which who how why does do did has have can could may might".split())
def toks(x): return [t for t in re.findall(r"[a-z0-9][a-z0-9_-]+",str(x).lower()) if t not in STOP]

def sparse_score(query, text):
    q=Counter(toks(query)); d=Counter(toks(text))
    if not q or not d: return 0.0
    return sum(min(q[t],d[t]) for t in q)/math.sqrt(sum(v*v for v in q.values())*sum(v*v for v in d.values()))

def pageindex_retrieve(query, top_k=10):
    rows=[]
    for root in trees.values():
        for n in root["children"]:
            path_text=" ".join(n["path"])
            score=0.35*sparse_score(query,path_text)+0.65*sparse_score(query,n["text"])
            rows.append({"passage_id":n["passage_id"],"node_id":n["node_id"],"score":score,"path":n["path"]})
    return pd.DataFrame(rows).sort_values(["score","node_id"],ascending=[False,True]).head(top_k).to_dict("records")

if len(questions_work):
    demo_pi=pageindex_retrieve(questions_work.iloc[0].question,top_k=5)
    display(pd.DataFrame(demo_pi))


# 11. Optional LLM tree-search adapter

A true reasoning-based PageIndex retrieval loop should let a model:

1. inspect the tree outline,
2. choose promising branches,
3. open only selected nodes,
4. repeat until evidence is sufficient,
5. return explicit node/page references.

We keep that behind an adapter so the notebook remains runnable without an API key. The output contract is deterministic and testable.


In [ ]:
def llm_pageindex_search(query, tree, llm=None, max_rounds=2, max_nodes_per_round=5):
    """Adapter contract. `llm` must implement `select_nodes(query, visible_nodes) -> list[node_id]`."""
    visible=[{"node_id":n["node_id"],"title":n["title"],"path":n["path"]} for n in tree["children"]]
    selected=[]
    rounds=[]
    if llm is None:
        # No-API fallback: deterministic hierarchical selector.
        ranked=sorted(tree["children"], key=lambda n:sparse_score(query," ".join(n["path"]) + " " + n["text"]), reverse=True)
        selected=[n["node_id"] for n in ranked[:max_nodes_per_round]]
        rounds.append({"round":1,"mode":"deterministic_fallback","selected":selected})
    else:
        for r in range(max_rounds):
            ids=llm.select_nodes(query,visible)
            ids=list(dict.fromkeys(ids))[:max_nodes_per_round]
            selected.extend(ids)
            rounds.append({"round":r+1,"mode":"llm","selected":ids})
            if not ids: break
            visible=[v for v in visible if v["node_id"] not in ids]
    selected=list(dict.fromkeys(selected))
    selected_nodes=[n for n in tree["children"] if n["node_id"] in selected]
    return {"selected_nodes":selected_nodes,"rounds":rounds,"passage_ids":[n["passage_id"] for n in selected_nodes]}


# 12. PageIndex evaluation

The key metric is not “tree nodes retrieved.” It is **gold evidence recovery**. We therefore map selected nodes back to canonical passage IDs and compute the same Hit@K and Recall@K metrics used by lexical/dense/hybrid retrieval.


In [ ]:
def pageindex_result(query, top_k=10):
    return {"passage_ids":[x["passage_id"] for x in pageindex_retrieve(query,top_k=top_k)],"seeds":[],"mode":"pageindex-style"}

pi_eval=eval_retrieval(questions_work, pageindex_result, k=10) if not questions_work.empty else pd.DataFrame()
if not pi_eval.empty:
    print("PageIndex-style deterministic baseline:",pi_eval[["hit","recall"]].mean(numeric_only=True).to_dict())


# 13. Compare flat retrieval vs Graph vs PageIndex

Notebook 07 established the flat retrieval family (BM25, dense, hybrid, reranker). This notebook adds two specialized representations:

- **GraphRAG:** relational / multi-hop retrieval.
- **PageIndex:** hierarchical / structure-first retrieval.

For a controlled benchmark, import Notebook 07 result tables when available. Otherwise, report only the channels that can be evaluated in this notebook.


In [ ]:
def summarize_metric(name, ev):
    if ev is None or ev.empty: return None
    return {"retriever":name,"Hit@10":ev.hit.mean(),"Recall@10":ev.recall.mean()}

rows=[]
for name,ev in [("Graph-1hop",eval_1 if 'eval_1' in globals() else pd.DataFrame()),("Graph-2hop",eval_2 if 'eval_2' in globals() else pd.DataFrame()),("PageIndex-style",pi_eval)]:
    s=summarize_metric(name,ev)
    if s: rows.append(s)
leaderboard=pd.DataFrame(rows)
display(leaderboard)


## 14. A more useful test: query-type routing

The architecture should not call graph traversal or PageIndex for every question. We build a transparent router prototype based on observable query signals:

- explicit biomedical entities
- relation words (`associated with`, `inhibits`, `causes`, etc.)
- comparison / compositional signals
- likely multi-hop wording

This is a **routing hypothesis**, not a learned policy. Notebook 09 will replace it with the bounded agentic retrieval orchestrator and measure routing regret.


In [ ]:
RELATION_WORDS={"associated","association","inhibits","inhibition","activates","treatment","treat","risk","causes","caused","expressed","interaction","mechanism","pathway"}
MULTIHOP_WORDS={"between","relationship","mechanism","through","via","mediated","link","connected","affects"}

def route_query(question):
    t=set(toks(question))
    entity_count=len(link_query_entities(question))
    relation_signal=len(t & RELATION_WORDS)
    multihop_signal=len(t & MULTIHOP_WORDS)
    if entity_count>=2 and (relation_signal+multihop_signal)>=1:
        route="graph"
    elif any(w in str(question).lower() for w in ["section", "guideline", "study design", "table", "figure", "in the paper", "according to"]):
        route="pageindex"
    else:
        route="flat"
    return {"route":route,"entity_count":entity_count,"relation_signal":relation_signal,"multihop_signal":multihop_signal}

route_df=pd.DataFrame([{"question_id":r.canonical_question_id,**route_query(r.question)} for _,r in questions_work.iterrows()]) if not questions_work.empty else pd.DataFrame()
display(route_df.route.value_counts(dropna=False).rename_axis("route").reset_index(name="questions"))


# 15. Graph path and provenance diagnostics

A graph retriever can look good on Recall@K while still returning weak evidence. We therefore inspect:

- number of graph paths explored,
- number of unique evidence passages,
- average supporting edges per returned passage,
- provenance completeness.


In [ ]:
graph_diag_rows=[]
for _,r in questions_work.head(100).iterrows():
    out=graph_retrieve(str(r.question),hops=2)
    support_hits=0
    for pid in out["passage_ids"]:
        support_hits += 1 if pid in set(passages_work[ID_COL].astype(str)) else 0
    graph_diag_rows.append({
        "question_id":r.canonical_question_id,
        "seed_entities":len(out["seeds"]),
        "visited_entities":len(out["visited_entities"]),
        "paths":len(out["paths"]),
        "returned_passages":len(out["passage_ids"]),
        "provenance_coverage":support_hits/max(1,len(out["passage_ids"])),
    })
graph_diag=pd.DataFrame(graph_diag_rows)
display(graph_diag.describe(include="all"))


# 16. Failure taxonomy for this notebook

Record failures with the same taxonomy used by the rest of BioRAG-X:

- **F03** entity extraction/linking
- **F09** graph linking / edge construction
- **F10** graph traversal
- **F11** PageIndex / hierarchy retrieval
- **F13** evidence incompleteness
- **F17** citation/provenance failure
- **F19** unnecessary retrieval

This lets Notebook 09 attribute recovery failures to the right subsystem rather than merely marking a question as “wrong.”


In [ ]:
def classify_graph_failure(question, result, gold_ids, k=10):
    pred=set(result.get("passage_ids",[])[:k])
    gold=set(map(str,gold_ids))
    if not result.get("seeds") and gold:
        return "F03_entity_linking"
    if pred & gold:
        return "OK"
    if result.get("visited_entities"):
        return "F10_graph_traversal"
    return "F09_graph_linking"


# 17. Export graph + PageIndex artifacts

The graph is exported in two forms:

1. **Edge evidence table** — relational and easy to audit.
2. **NetworkX adjacency pickle** — convenient for local traversal experiments.

The PageIndex-style tree is exported as JSON so later notebooks can reuse it without rebuilding structure.


In [ ]:
edges_path=ARTIFACT_DIR/"graph_edge_evidence.parquet"
entities_path=ARTIFACT_DIR/"graph_entities.parquet"
tree_path=ARTIFACT_DIR/"pageindex_style_trees.json"
manifest_path=ARTIFACT_DIR/"graph_pageindex_manifest.json"

if not edges_evidence_df.empty:
    edges_evidence_df.to_parquet(edges_path,index=False)
entities_df.to_parquet(entities_path,index=False)
tree_path.write_text(json.dumps(trees,indent=2,ensure_ascii=False),encoding="utf-8")

manifest={
    "project":"BioRAG-X",
    "pipeline":"08_graph_and_pageindex",
    "seed":SEED,
    "benchmark_questions":len(questions_work),
    "passages_used":len(passages_work),
    "graph_nodes":G.number_of_nodes(),
    "graph_edges":G.number_of_edges(),
    "provenance_edge_coverage": float(np.mean([bool(d.get("supporting_passage_ids")) for _,_,d in G.edges(data=True)])) if G.number_of_edges() else 1.0,
    "artifacts":{
        "graph_edge_evidence":str(edges_path),
        "graph_entities":str(entities_path),
        "pageindex_style_trees":str(tree_path),
    },
    "notes":[
        "Graph extraction is a transparent baseline, not a biomedical gold-standard extractor.",
        "PageIndex evaluation is PageIndex-style/benchmark-harness behavior unless the official engine is explicitly integrated.",
        "Every graph edge must preserve passage provenance.",
    ],
}
manifest_path.write_text(json.dumps(manifest,indent=2),encoding="utf-8")
print(manifest_path)


# 18. Research conclusions to carry into Notebook 09

Before the agentic retrieval orchestrator is allowed to route work dynamically, we need answers to four empirical questions:

### A. Does graph traversal add evidence that flat retrieval misses?
Measure **Graph Recall@K − Hybrid Recall@K** on multi-hop / multi-passage subsets.

### B. Is 2-hop actually worth it?
Measure **2-hop gain per additional traversal cost**. If 2-hop barely improves recall, keep the budget at 1 hop.

### C. Does hierarchy recover evidence that similarity misses?
Compare PageIndex-style retrieval with BM25/dense/hybrid on structurally dependent questions.

### D. Can we route only the questions that benefit?
Notebook 09 should learn or calibrate a policy that decides among:
`flat → graph → pageindex → expansion → decomposition`
while minimizing **routing regret, unnecessary retrieval, and latency**.

The winning architecture is therefore not “GraphRAG + PageIndex + Dense.” It is the **smallest retrieval computation that obtains sufficient, attributable evidence for the current question**.


## Handoff to Notebook 09 — Agentic Retrieval

Notebook 09 will consume:

- canonical passages/questions/gold relationships from Notebook 02,
- dense/ANN artifacts from Notebook 06,
- hybrid/reranker outputs from Notebook 07,
- graph + provenance artifacts from this notebook,
- PageIndex-style tree artifacts from this notebook.

It will implement the bounded state machine:

`ANALYZE → ROUTE → RETRIEVE → FUSE → RERANK → ASSESS → STOP / RECOVER`

with recovery actions including query rewrite, HyDE, Query2Doc, graph traversal, PageIndex tree search, and decomposition.

**Target decisions:** retrieval decision accuracy, routing regret, recovery success rate, help/harm rate, unnecessary retrieval rate, and agent efficiency.
